# EvalHub - Check Evaluation Status

This notebook demonstrates how to check the status of evaluation jobs (running and completed) using the EvalHub REST API.

## Prerequisites
- EvalHub deployed and accessible
- At least one evaluation job created (use `01-create-evaluation.ipynb`)

In [ ]:
import requests
import json
import os
from datetime import datetime
from urllib3.exceptions import InsecureRequestWarning
import warnings
import time

# Suppress SSL warnings for demo purposes
warnings.simplefilter('ignore', InsecureRequestWarning)

## Configuration

In [ ]:
# EvalHub API endpoint
EVALHUB_URL = "https://evalhub-redhat-ods-applications.apps.ocp.sandbox5548.opentlc.com"

# Authentication
OCP_TOKEN = os.getenv('OPENSHIFT_TOKEN', 'sha256~YOUR_TOKEN_HERE')

# Request headers
headers = {
    "Authorization": f"Bearer {OCP_TOKEN}",
    "Content-Type": "application/json"
}

# Tenant/namespace to check
TENANT = "models-as-a-service"

print(f"✓ EvalHub URL: {EVALHUB_URL}")
print(f"✓ Tenant: {TENANT}")

## List All Evaluation Jobs

In [ ]:
response = requests.get(
    f"{EVALHUB_URL}/api/v1/evaluations/jobs",
    headers=headers,
    params={"tenant": TENANT},
    verify=False
)

print(f"Status Code: {response.status_code}\n")

if response.status_code == 200:
    jobs = response.json()
    
    if isinstance(jobs, dict) and 'jobs' in jobs:
        jobs_list = jobs['jobs']
    elif isinstance(jobs, list):
        jobs_list = jobs
    else:
        jobs_list = []
    
    if not jobs_list:
        print("No evaluation jobs found.")
    else:
        print(f"Found {len(jobs_list)} evaluation job(s):\n")
        print("-" * 100)
        
        for i, job in enumerate(jobs_list, 1):
            print(f"\n{i}. {job.get('name', 'Unnamed')}")
            print(f"   ID: {job.get('id', job.get('job_id', 'N/A'))}")
            print(f"   Status: {job.get('status', 'unknown')}")
            print(f"   Evaluation: {job.get('evaluation', 'N/A')}")
            print(f"   Provider: {job.get('provider', 'N/A')}")
            print(f"   Model: {job.get('model', {}).get('name', 'N/A')}")
            print(f"   Created: {job.get('created_at', 'N/A')}")
            print(f"   Updated: {job.get('updated_at', 'N/A')}")
            
            if job.get('status') == 'completed':
                print(f"   ✓ Result: {job.get('result', 'N/A')}")
            elif job.get('status') == 'failed':
                print(f"   ✗ Error: {job.get('error', 'N/A')}")
            elif job.get('status') == 'running':
                print(f"   ⟳ Progress: {job.get('progress', 'N/A')}")
        
        print("\n" + "-" * 100)
else:
    print(f"Error: {response.text}")

## Get Specific Job Details

Check the status of a specific job by ID

In [ ]:
# Try to load job ID from previous run
job_id = None
if os.path.exists('evalhub_job_id.txt'):
    with open('evalhub_job_id.txt', 'r') as f:
        job_id = f.read().strip()
    print(f"Loaded job ID from file: {job_id}")
else:
    # Manually set job ID if file doesn't exist
    job_id = input("Enter Job ID: ").strip()

if job_id:
    response = requests.get(
        f"{EVALHUB_URL}/api/v1/evaluations/jobs/{job_id}",
        headers=headers,
        verify=False
    )
    
    print(f"\nStatus Code: {response.status_code}\n")
    
    if response.status_code == 200:
        job = response.json()
        print("Job Details:")
        print(json.dumps(job, indent=2))
    else:
        print(f"Error: {response.text}")
else:
    print("No job ID provided.")

## Filter Jobs by Status

Display running, pending, completed, and failed jobs separately

In [ ]:
response = requests.get(
    f"{EVALHUB_URL}/api/v1/evaluations/jobs",
    headers=headers,
    params={"tenant": TENANT},
    verify=False
)

if response.status_code == 200:
    jobs = response.json()
    
    if isinstance(jobs, dict) and 'jobs' in jobs:
        jobs_list = jobs['jobs']
    elif isinstance(jobs, list):
        jobs_list = jobs
    else:
        jobs_list = []
    
    # Group by status
    running = [j for j in jobs_list if j.get('status') == 'running']
    pending = [j for j in jobs_list if j.get('status') == 'pending']
    completed = [j for j in jobs_list if j.get('status') == 'completed']
    failed = [j for j in jobs_list if j.get('status') == 'failed']
    
    print(f"\n📊 Job Summary:")
    print(f"  ⟳ Running:   {len(running)}")
    print(f"  ⏸ Pending:   {len(pending)}")
    print(f"  ✓ Completed: {len(completed)}")
    print(f"  ✗ Failed:    {len(failed)}")
    print(f"  ━ Total:     {len(jobs_list)}\n")
    
    # Show running jobs
    if running:
        print("\n⟳ RUNNING JOBS:")
        print("-" * 80)
        for job in running:
            print(f"  • {job.get('name', 'Unnamed')} ({job.get('id')})")
            print(f"    Evaluation: {job.get('evaluation')} | Model: {job.get('model', {}).get('name')}")
            print(f"    Progress: {job.get('progress', 'N/A')}\n")
    
    # Show pending jobs
    if pending:
        print("\n⏸ PENDING JOBS:")
        print("-" * 80)
        for job in pending:
            print(f"  • {job.get('name', 'Unnamed')} ({job.get('id')})")
            print(f"    Evaluation: {job.get('evaluation')} | Model: {job.get('model', {}).get('name')}")
            print(f"    Created: {job.get('created_at', 'N/A')}\n")
    
    # Show completed jobs
    if completed:
        print("\n✓ COMPLETED JOBS:")
        print("-" * 80)
        for job in completed:
            print(f"  • {job.get('name', 'Unnamed')} ({job.get('id')})")
            print(f"    Evaluation: {job.get('evaluation')} | Model: {job.get('model', {}).get('name')}")
            print(f"    Result: {job.get('result', 'N/A')}")
            print(f"    Completed: {job.get('updated_at', 'N/A')}\n")
    
    # Show failed jobs
    if failed:
        print("\n✗ FAILED JOBS:")
        print("-" * 80)
        for job in failed:
            print(f"  • {job.get('name', 'Unnamed')} ({job.get('id')})")
            print(f"    Evaluation: {job.get('evaluation')} | Model: {job.get('model', {}).get('name')}")
            print(f"    Error: {job.get('error', 'N/A')}")
            print(f"    Failed: {job.get('updated_at', 'N/A')}\n")
else:
    print(f"Error: {response.text}")

## Monitor Job Progress

Poll a specific job until it completes (or fails)

In [ ]:
# Load or specify job ID to monitor
job_id = None
if os.path.exists('evalhub_job_id.txt'):
    with open('evalhub_job_id.txt', 'r') as f:
        job_id = f.read().strip()
else:
    job_id = input("Enter Job ID to monitor: ").strip()

if job_id:
    print(f"Monitoring job: {job_id}")
    print("Press Ctrl+C to stop monitoring\n")
    
    max_iterations = 60  # Poll for up to 5 minutes (60 * 5 seconds)
    iteration = 0
    
    while iteration < max_iterations:
        try:
            response = requests.get(
                f"{EVALHUB_URL}/api/v1/evaluations/jobs/{job_id}",
                headers=headers,
                verify=False
            )
            
            if response.status_code == 200:
                job = response.json()
                status = job.get('status', 'unknown')
                timestamp = datetime.now().strftime('%H:%M:%S')
                
                if status == 'running':
                    print(f"[{timestamp}] ⟳ Status: {status} | Progress: {job.get('progress', 'N/A')}")
                elif status == 'pending':
                    print(f"[{timestamp}] ⏸ Status: {status} | Waiting to start...")
                elif status == 'completed':
                    print(f"[{timestamp}] ✓ Status: {status}")
                    print(f"\nResult: {job.get('result', 'N/A')}")
                    print("\nJob completed successfully!")
                    break
                elif status == 'failed':
                    print(f"[{timestamp}] ✗ Status: {status}")
                    print(f"\nError: {job.get('error', 'N/A')}")
                    print("\nJob failed.")
                    break
                else:
                    print(f"[{timestamp}] Status: {status}")
                
                time.sleep(5)  # Poll every 5 seconds
                iteration += 1
            else:
                print(f"Error fetching job status: {response.status_code}")
                print(response.text)
                break
                
        except KeyboardInterrupt:
            print("\n\nMonitoring stopped by user.")
            break
    
    if iteration >= max_iterations:
        print("\nReached maximum polling iterations. Job still in progress.")
else:
    print("No job ID provided.")

## Check Kubernetes Resources

Verify the underlying Kubernetes Job and Pod status

In [ ]:
# This cell requires kubectl to be available
import subprocess

print("Kubernetes Jobs in models-as-a-service namespace:\n")
result = subprocess.run(
    ["kubectl", "get", "jobs", "-n", "models-as-a-service"],
    capture_output=True,
    text=True
)
print(result.stdout)

print("\nKubernetes Pods (evaluation jobs):\n")
result = subprocess.run(
    ["kubectl", "get", "pods", "-n", "models-as-a-service", "--selector=job-name"],
    capture_output=True,
    text=True
)
print(result.stdout)

## View MLflow Experiments

EvalHub stores evaluation results in MLflow. Check the MLflow dashboard or use the API:

In [ ]:
mlflow_url = "https://mlflow-redhat-ods-applications.apps.ocp.sandbox5548.opentlc.com"
print(f"MLflow Dashboard: {mlflow_url}")
print(f"\nOpen the dashboard to view evaluation experiment results.")
print(f"Look for experiments in the '{TENANT}' namespace.")

## Troubleshooting

### Common Issues:

1. **No jobs returned**: Verify the tenant/namespace is correct
2. **Job stuck in Pending**: Check pod events with `kubectl describe pod -n models-as-a-service`
3. **Job failed**: Check pod logs with `kubectl logs -n models-as-a-service <pod-name>`
4. **403 errors**: Ensure you have MLflow RBAC permissions (see `13-mlflow/RoleBinding-evalhub-mlflow-user-access.yaml`)

### Known RHOAI 3.4.0 Issues:

- **Bug**: EvalHub may incorrectly use the model API key token value as a Kubernetes secret name, causing jobs to fail with "secret not found" errors
- **Workaround**: Report the issue to Red Hat support and use models without authentication for testing